In [1]:
import pandas as pd
import random
from math import radians, sin, cos, atan2, sqrt
from ipyleaflet import Map, Marker, Polyline, basemaps, Icon, Circle
from ipywidgets import HTML
from IPython.display import display
from datetime import datetime
import asyncio
import ipywidgets as widgets

# Variables globales de mapa
m = None
ruta = None
marker_rescatista = None
marker_victima = None
circulo_victima = None

# Variables globales del Panel de Telemetría
lbl_distancia = None
lbl_velocidad = None
lbl_ritmo = None
lbl_tiempo = None
lbl_pasos = None
lbl_eta = None
lbl_dist_restante = None
lbl_punto = None
lbl_modo = None

In [2]:
# =======================================================
#   CELDA 2 — Funciones + Simulación ASYNC + Excel
# =======================================================

import asyncio
import pandas as pd
import random
from math import radians, sin, cos, atan2, sqrt
from datetime import datetime
from ipyleaflet import Marker, Icon
from ipywidgets import HTML

# =======================================================
#                   FUNCIONES BASE
# =======================================================

def haversine_m(lat1, lon1, lat2, lon2):
    """Distancia entre dos puntos GPS en METROS."""
    R = 6371000
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c


def mover_rescatista(lat, lon, lat_dest, lon_dest, distancia):
    """Movimiento directo hacia la víctima."""
    d_total = haversine_m(lat, lon, lat_dest, lon_dest)
    if d_total == 0:
        return lat, lon
    f = distancia / d_total
    return lat + (lat_dest - lat) * f, lon + (lon_dest - lon) * f


def mover_rescatista_realista(lat, lon, lat_dest, lon_dest, distancia):
    """Movimiento humano realista (zig-zag)."""
    lat1, lon1 = radians(lat), radians(lon)
    lat2, lon2 = radians(lat_dest), radians(lon_dest)

    dlon = lon2 - lon1
    x = cos(lat2) * sin(dlon)
    y = cos(lat1)*sin(lat2) - sin(lat1)*cos(lat2)*cos(dlon)
    angulo = atan2(x, y)

    desviacion = random.uniform(-0.45, 0.45)
    angulo += desviacion

    dlat = distancia * cos(angulo) / 111320
    dlon = distancia * sin(angulo) / (111320 * cos(lat))

    return lat + dlat, lon + dlon


# =======================================================
#            SIMULACIÓN PRINCIPAL ASYNC
# =======================================================

async def simular_rescate(modo="zigzag"):
    """
    Simulación en tiempo real usando asyncio.
    """

    global m, ruta, marker_rescatista, marker_victima
    global lbl_distancia, lbl_velocidad, lbl_ritmo, lbl_tiempo
    global lbl_pasos, lbl_eta, lbl_dist_restante, lbl_punto, lbl_modo, inicio_tiempo

    print(f"\n🔄 Simulación en modo {modo.upper()}...\n")

    inicio_tiempo = datetime.now()

    # LIMPIAR RUTA PERO NO BORRAR INICIO/FINAL
    ruta.locations = []

    layers_to_keep = {m.layers[0], ruta, marker_rescatista, marker_victima}
    for lyr in list(m.layers):
        if lyr not in layers_to_keep:
            m.remove_layer(lyr)

    # Parámetros de paso
    paso_m = 0.75
    pasos_min, pasos_max = 8, 20
    distancia_objetivo = 6

    lat_r, lon_r = rescuer_lat, rescuer_lon
    dist_acum = 0.0
    punto_id = 0

    # =======================================================
    #                MARCADOR DE PUNTO 0
    # =======================================================
    hora0 = datetime.now().strftime("%H:%M:%S")
    dist0 = haversine_m(lat_r, lon_r, victim_lat, victim_lon)

    popup0 = HTML(
        value=f"""
        <b>Punto 0 (Inicio)</b><br><hr>
        🕒 {hora0}<br>
        📍 {lat_r:.6f}, {lon_r:.6f}<br><hr>
        📏 Δ: 0.00 m<br>
        📏 Total: 0.00 m<br>
        🎯 A víctima: {dist0:.2f} m<br>
        """
    )

    inicio_marker = Marker(location=(lat_r, lon_r))
    inicio_marker.popup = popup0
    m.add_layer(inicio_marker)

    registros = []
    registros.append({
        "Punto": 0,
        "Hora": hora0,
        "Latitud": lat_r,
        "Longitud": lon_r,
        "Distancia_delta_m": 0,
        "Distancia_total_m": 0,
        "Distancia_a_victima_m": dist0,
        "Pasos": 0,
        "Avance_estimado_m": 0,
        "Modo": modo
    })

    # =======================================================
    #                 BUCLE PRINCIPAL
    # =======================================================

    for _ in range(600):

        dist_restante = haversine_m(lat_r, lon_r, victim_lat, victim_lon)

        if dist_restante <= distancia_objetivo:
            print("🎯 Rescatista llegó a la víctima.")
            break

        # TIEMPOS (REALISTA)
        intervalo_logico = 30  # Solo mantener 30 segundos para el intervalo lógico
        intervalo_real = intervalo_logico / 30  # 1 segundo lógico = 30 segundos reales

        pasos = random.randint(pasos_min, pasos_max)
        avance_m = pasos * paso_m

        if modo == "directo":
            lat_n, lon_n = mover_rescatista(lat_r, lon_r, victim_lat, victim_lon, avance_m)
        elif modo == "descanso":
            if random.random() < 0.25:
                await asyncio.sleep(intervalo_real)
                continue
            lat_n, lon_n = mover_rescatista_realista(lat_r, lon_r, victim_lat, victim_lon, avance_m)
        else:
            lat_n, lon_n = mover_rescatista_realista(lat_r, lon_r, victim_lat, victim_lon, avance_m)

        distancia_delta = haversine_m(lat_r, lon_r, lat_n, lon_n)
        dist_acum += distancia_delta

        lat_r, lon_r = lat_n, lon_n
        punto_id += 1

        # =======================================================
        #                MARCADOR DEL PUNTO
        # =======================================================
        icono_p = Icon(
            icon_url="https://raw.githubusercontent.com/pointhi/leaflet-color-markers/master/img/marker-icon-blue.png",
            shadow_url="https://cdnjs.cloudflare.com/ajax/libs/leaflet/0.7.7/images/marker-shadow.png",
            icon_size=[25, 41],
            icon_anchor=[12, 41]
        )

        punto_marker = Marker(location=(lat_r, lon_r), icon=icono_p)

        popup = HTML(
            value=f"""
            <b>Punto {punto_id}</b><br><hr>
            🕒 {datetime.now().strftime("%H:%M:%S")}<br>
            📍 {lat_r:.6f}, {lon_r:.6f}<br><hr>
            📏 Δ: {distancia_delta:.2f} m<br>
            📏 Total: {dist_acum:.2f} m<br>
            🎯 A víctima: {dist_restante:.2f} m<br><hr>
            🦶 Pasos: {pasos}<br>
            🚶 Avance: {avance_m:.2f} m<br>
            ⏱ Intervalo: {intervalo_logico}s ({intervalo_real:.1f}s real)<br>
            🟪 Modo: {modo.upper()}
            """
        )
        punto_marker.popup = popup
        m.add_layer(punto_marker)

        # =======================================================
        # ACTUALIZAR RUTA, MARCADOR Y TELEMETRÍA
        # =======================================================
        ruta.locations = ruta.locations + [(lat_r, lon_r)]
        marker_rescatista.location = (lat_r, lon_r)

        tiempo_total = (datetime.now() - inicio_tiempo).seconds

        lbl_distancia.value = f"<b>{dist_acum:.2f} m</b>"
        lbl_dist_restante.value = f"<b>{dist_restante:.2f} m</b>"
        lbl_punto.value = f"<b>Punto {punto_id}</b>"
        lbl_modo.value = f"<b>{modo.upper()}</b>"
        lbl_tiempo.value = f"<b>{tiempo_total//60:02}:{tiempo_total%60:02}</b>"

        velocidad = distancia_delta / intervalo_real
        lbl_velocidad.value = f"<b>{velocidad:.2f} m/s</b><br><span style='font-size:11px'>{velocidad*3.6:.2f} km/h</span>"

        if velocidad > 0:
            lbl_ritmo.value = f"<b>{(1000/velocidad)/60:.2f}</b><br><span style='font-size:11px;'>min/km</span>"
        else:
            lbl_ritmo.value = "--"

        lbl_pasos.value = f"<b>{pasos}</b><br><span style='font-size:11px;'>pasos</span>"

        if velocidad > 0:
            eta = dist_restante / velocidad
            lbl_eta.value = f"<b>{eta:.0f}s</b><br><span style='font-size:11px'>ETA</span>"
        else:
            lbl_eta.value = "--"

        # Registrar datos
        registros.append({
            "Punto": punto_id,
            "Hora": datetime.now().strftime("%H:%M:%S"),
            "Latitud": lat_r,
            "Longitud": lon_r,
            "Distancia_delta_m": round(distancia_delta, 2),
            "Distancia_total_m": round(dist_acum, 2),
            "Distancia_a_victima_m": round(dist_restante, 2),
            "Pasos": pasos,
            "Avance_estimado_m": round(avance_m, 2),
            "Modo": modo
        })

        # Delay real
        await asyncio.sleep(intervalo_real)

    # =======================================================
    #     GUARDAR EN EXCEL
    # =======================================================
    df = pd.DataFrame(registros)
    archivo = "simulacion_rescate.xlsx"

    try:
        with pd.ExcelWriter(archivo, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
            df.to_excel(writer, sheet_name=modo.upper(), index=False)
        print(f"📘 Datos guardados en hoja: {modo.upper()}")
    except FileNotFoundError:
        with pd.ExcelWriter(archivo, engine="openpyxl") as writer:
            df.to_excel(writer, sheet_name=modo.upper(), index=False)
        print(f"📘 Archivo creado y hoja agregada: {modo.upper()}")

    return df


In [3]:
global lbl_distancia, lbl_velocidad, lbl_ritmo, lbl_tiempo
global lbl_pasos, lbl_eta, lbl_dist_restante, lbl_punto, lbl_modo

lbl_distancia = widgets.HTML("<b>0.0 m</b>")
lbl_velocidad = widgets.HTML("<b>0.00 m/s</b><br><span style='font-size:11px;'>0.00 km/h</span>")
lbl_ritmo = widgets.HTML("<b>--</b><br><span style='font-size:11px;'>min/km</span>")
lbl_tiempo = widgets.HTML("<b>00:00</b>")
lbl_pasos = widgets.HTML("<b>0</b><br><span style='font-size:11px;'>pasos</span>")
lbl_eta = widgets.HTML("<b>--</b><br><span style='font-size:11px;'>ETA</span>")
lbl_dist_restante = widgets.HTML("<b>0.0 m</b>")
lbl_punto = widgets.HTML("<b>Punto 0</b>")
lbl_modo = widgets.HTML("<b>---</b>")

panel = widgets.VBox([
    widgets.HTML("<h4>📊 Panel de Telemetría</h4>"),
    widgets.HBox([widgets.VBox([widgets.HTML("Distancia total"), lbl_distancia]),
                  widgets.VBox([widgets.HTML("Velocidad"), lbl_velocidad])]),
    widgets.HBox([widgets.VBox([widgets.HTML("Ritmo"), lbl_ritmo]),
                  widgets.VBox([widgets.HTML("Tiempo"), lbl_tiempo])]),
    widgets.HBox([widgets.VBox([widgets.HTML("Pasos / Cadencia"), lbl_pasos]),
                  widgets.VBox([widgets.HTML("ETA estimado"), lbl_eta])]),
    widgets.HBox([widgets.VBox([widgets.HTML("Distancia restante"), lbl_dist_restante]),
                  widgets.VBox([widgets.HTML("Punto actual"), lbl_punto])]),
    widgets.VBox([widgets.HTML("Modo"), lbl_modo])
])

display(panel)


In [4]:
# ============================================
#   CELDA 4 — MAPA INTERACTIVO (OpenStreetMap)
# ============================================

# Coordenadas del rescatista 
rescuer_lat = 0.363209
rescuer_lon = -78.110327

# Coordenadas de la víctima
victim_lat = 0.361200
victim_lon = -78.109157

# Reset de valores globales
m = None
ruta = None
marker_rescatista = None
marker_victima = None
circulo_victima = None


# --------------------------------------------
# Crear MAPA
# --------------------------------------------
m = Map(
    center=(rescuer_lat, rescuer_lon),
    zoom=17,
    basemap=basemaps.OpenStreetMap.Mapnik,
    scroll_wheel_zoom=True
)

# --------------------------------------------
# Crear polilínea para la ruta
# --------------------------------------------
ruta = Polyline(locations=[], color="blue", weight=4)
m.add_layer(ruta)

# --------------------------------------------
# Marcador del rescatista (INICIO)
# --------------------------------------------
icono_inicio = Icon(
    icon_url="https://raw.githubusercontent.com/pointhi/leaflet-color-markers/master/img/marker-icon-blue.png",
    shadow_url="https://cdnjs.cloudflare.com/ajax/libs/leaflet/0.7.7/images/marker-shadow.png",
    icon_size=[25, 41],
    icon_anchor=[12, 41]
)

marker_rescatista = Marker(
    location=(rescuer_lat, rescuer_lon),
    icon=icono_inicio
)
marker_rescatista.popup = HTML("<b>📍 Punto de Inicio (Rescatista)</b>")
m.add_layer(marker_rescatista)

# --------------------------------------------
# Marcador de la víctima (FINAL)
# --------------------------------------------
icono_final = Icon(
    icon_url="https://raw.githubusercontent.com/pointhi/leaflet-color-markers/master/img/marker-icon-red.png",
    shadow_url="https://cdnjs.cloudflare.com/ajax/libs/leaflet/0.7.7/images/marker-shadow.png",
    icon_size=[25, 41],
    icon_anchor=[12, 41]
)

marker_victima = Marker(
    location=(victim_lat, victim_lon),
    icon=icono_final
)
marker_victima.popup = HTML("<b>🎯 Objetivo del rescate</b>")
m.add_layer(marker_victima)

# --------------------------------------------
# Círculo indicando zona de llegada (6 m)
# --------------------------------------------
circulo_victima = Circle(
    location=(victim_lat, victim_lon),
    radius=6,
    color="red",
    fill_color="red",
    fill_opacity=0.15
)
m.add_layer(circulo_victima)

# Ajustar mapa para mostrar inicio y final
m.fit_bounds([(rescuer_lat, rescuer_lon), (victim_lat, victim_lon)])

display(m)


Map(center=[0.363209, -78.110327], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title',…

In [5]:
# =============================================
#     CELDA 5 — Botones de control de modos
# =============================================

btn_directo = widgets.Button(
    description="🚶 Modo Directo",
    button_style='info',
    layout=widgets.Layout(width="160px")
)

btn_zigzag = widgets.Button(
    description="〰️ Modo Zigzag",
    button_style='warning',
    layout=widgets.Layout(width="160px")
)

btn_descanso = widgets.Button(
    description="😴 Modo Descanso",
    button_style='danger',
    layout=widgets.Layout(width="160px")
)

estado = widgets.HTML(
    value="<b>Seleccione un modo para iniciar la simulación.</b>"
)

# ==================================================
#   VERSIONES ASYNC — EJECUTAN LA SIMULACIÓN REAL
# ==================================================

async def ejecutar_directo(_):
    estado.value = "⏳ Ejecutando → MODO DIRECTO…"
    await asyncio.sleep(0.1)
    await simular_rescate("directo")
    estado.value = "✔ Simulación DIRECTA finalizada."

async def ejecutar_zigzag(_):
    estado.value = "⏳ Ejecutando → MODO ZIGZAG…"
    await asyncio.sleep(0.1)
    await simular_rescate("zigzag")
    estado.value = "✔ Simulación ZIGZAG finalizada."

async def ejecutar_descansos(_):
    estado.value = "⏳ Ejecutando → MODO DESCANSO…"
    await asyncio.sleep(0.1)
    await simular_rescate("descanso")
    estado.value = "✔ Simulación con DESCANSOS finalizada."

# ==================================================
#     Conexión de Botones → Funciones ASYNC
# ==================================================

btn_directo.on_click(lambda btn: asyncio.ensure_future(ejecutar_directo(btn)))
btn_zigzag.on_click(lambda btn: asyncio.ensure_future(ejecutar_zigzag(btn)))
btn_descanso.on_click(lambda btn: asyncio.ensure_future(ejecutar_descansos(btn)))

# Mostrar panel de botones
display(widgets.VBox([
    widgets.HBox([btn_directo, btn_zigzag, btn_descanso]),
    estado
]))
